# Finalising dataframes


In [1]:
import pandas as pd
from datetime import datetime

## Monthly data

In [15]:
#load excess data 
data_ex = pd.read_csv('../data/intermediate/excess.csv')
data_ex = data_ex[data_ex["Year"] == 2022]
data_ex = data_ex[['Region',  'Excessive Marriages', 'ex_nup_per100k']]

#load regional data
data_se= pd.read_csv('../data/intermediate/soc_econ_data.csv')
data_se = data_se.drop(columns=["Unnamed: 0"])
data_se = data_se[data_se["year"] > 2021].groupby('Region').mean().reset_index()
data_se = data_se[['Region','population', 'urban_share', 'disability_per10k',
       'uneployment', 'median_income', 'consumption', 'av_income',
       'share_poverty', '% Russians', '% Slavs', 'share_of_total',
       'share_of_income']]

#merge excess and regional
region_name_mapping = {
    'Архангельская область': 'Архангельская область без АО',
    'Тюменская область': 'Тюменская область без АО'
}

data_se['Region'] = data_se['Region'].replace(region_name_mapping)
data_cs = data_se.merge(data_ex, on = 'Region', how = 'left')

#load data on governors
data_gov = pd.read_csv('../data/gov.csv')
today = pd.to_datetime(datetime.today().date())
data_gov['end'] = data_gov['end'].fillna(today)

#load monthly casualties 
df_monthly = pd.read_csv('../data/daily.csv')
df_monthly = df_monthly[['region', 'branch', 'name','slavic','death_month']]
df_monthly = df_monthly.rename(columns={df_monthly.columns[0] : "Region"})

#load monthly deposits
dep = pd.read_excel("../data/deposits.xlsx", header = 1)[:95]
dep = dep.rename(columns={dep.columns[0] : "Region"})
dep_long = pd.melt(dep, id_vars=['Region'],
                   var_name='Date', 
                   value_name='Deposits') 



In [3]:
#regional names unification

region_name_mapping = {
    'Архангельская область без данных по Ненецкому автономному округу': 'Архангельская область без АО',
    'в том числе Ненецкий автономный округ': 'Ненецкий АО',
    'Кемеровская область - Кузбасс': 'Кемеровская область',
    'Город Москва столица Российской Федерации город федерального значения': 'Москва',
    'Город Санкт-Петербург город федерального значения': 'Санкт-Петербург',
    'Город федерального значения Севастополь': 'Севастополь',
    'Еврейская автономная область': 'Еврейская АО',
    'Кабардино-Балкарская Республика': 'Кабардино-Балкария',
    'Карачаево-Черкесская Республика': 'Карачаево-Черкесия',
    'Республика Адыгея (Адыгея)': 'Республика Адыгея',
    'Республика Саха (Якутия)': 'Якутия',
    'Республика Северная Осетия - Алания': 'Северная Осетия',
    'Республика Татарстан (Татарстан)': 'Республика Татарстан',
    'Чувашская Республика - Чувашия': 'Чувашская Республика',
    'Тюменская область без данных по Ханты-Мансийскому автономному округу - Югре и Ямало-Ненецкому автономному округу': 'Тюменская область без АО',
    'в том числе Ханты-Мансийский автономный округ - Югра': 'Ханты-Мансийский АО',
    'в том числе Ямало-Ненецкий автономный округ': 'Ямало-Hенецкий АО',
    'Чукотский автономный округ': 'Чукотский АО',
    'г. Москва' : 'Москва',
    'г. Санкт-Петербург' : 'Санкт-Петербург',
    'г. Севастополь' : 'Севастополь'
}

dep_long['Region'] = dep_long['Region'].replace(region_name_mapping)

In [4]:
#regional names unification

region_name_mapping = {
    'Архангельская область': 'Архангельская область без АО',
    'Ненецкий автономный округ': 'Ненецкий АО',
    'Еврейская автономная область': 'Еврейская АО',
    'Кабардино-Балкарская Республика': 'Кабардино-Балкария',
    'Республика Карачаево-Черкесия': 'Карачаево-Черкесия',
    'Республика Саха (Якутия)': 'Якутия',
    'Республика Северная Осетия-Алания': 'Северная Осетия',
    'Тюменская область': 'Тюменская область без АО',
    'Ханты-Мансийский автономный округ - Югра': 'Ханты-Мансийский АО',
    'Ямало-Ненецкий автономный округ': 'Ямало-Hенецкий АО',
    'Чукотский автономный округ': 'Чукотский АО',
}

df_monthly['Region'] = df_monthly['Region'].replace(region_name_mapping)


In [5]:
#breaking down casualties by branch

contract = ['автомобильные', 'артиллерия', 'ВДВ', 'военмед', 'военные пилоты',
       'войска связи', 'войсковая ПВО', 'инженерные войска', 'МВД','морпехи', 'моряки',
       'мотострелковые войска', 'наземные авиаслужбы', 'нацгвардия',
       'РХБЗ', 'спецназ', 'танковые войска', 'ФСБ', 
       'другие войска', 'ЖД', 'военная полиция', 'СК', 'ФСО']

df_monthly["total"] = 1
df_monthly["drafted"] = df_monthly["pmc"] = df_monthly["volunteers"] = df_monthly["prisoners"] = df_monthly['contract'] = 0
df_monthly.loc[df_monthly['branch'] == 'добровольцы', 'volunteers'] = 1
df_monthly.loc[df_monthly['branch'] == 'мобилизованные', 'drafted'] = 1
df_monthly.loc[df_monthly['branch'] == 'ЧВК', 'pmc'] = 1
df_monthly.loc[df_monthly['branch'] == 'заключенные', 'prisoners'] = 1
df_monthly.loc[df_monthly['branch'].isin(contract), 'contract'] = 1

In [6]:
#date format unification

df_monthly['death_month'] = pd.to_datetime(df_monthly['death_month'].astype(str), format='%m.%Y', errors ='coerce')
dep_long['Date'] = pd.to_datetime(dep_long['Date'], format='%d.%m.%Y', errors='coerce')
data_gov['start'] = pd.to_datetime(data_gov['start'], errors='coerce')
data_gov['end'] = pd.to_datetime(data_gov['end'], errors='coerce')


df_agg = df_monthly.dropna(subset = ['death_month']).groupby(['Region', 'death_month']).agg(
    slavic_name = ('slavic', lambda x: (x == 1).sum()),
    non_slavic_name = ('slavic', lambda x: (x == 0).sum()),
    pmc = ('pmc', lambda x: (x == 1).sum()),
    drafted = ('drafted', lambda x: (x == 1).sum()),
    volunteers = ('volunteers', lambda x: (x == 1).sum()),
    prisoners = ('prisoners', lambda x: (x == 1).sum()),
    contract = ('contract', lambda x: (x == 1).sum()),
    total = ('total', lambda x: (x == 1).sum())).reset_index() 


In [7]:
all_regions = df_monthly['Region'].unique()
all_months = pd.date_range(
    start=df_monthly['death_month'].min(),
    end=df_monthly['death_month'].max(),
    freq='MS' 
)

complete_grid = pd.MultiIndex.from_product(
    [all_regions, all_months],
    names=['Region', 'death_month']
).to_frame(index=False)


In [8]:
final_result = (
    complete_grid.merge(
        df_agg,
        on=['Region', 'death_month'],
        how='left'
    )
    .fillna({'slavic_name': 0, 'non_slavic_name': 0})  
    .sort_values(['Region', 'death_month'])
)
final_result[['slavic_cumulative', 'non_slavic_cumulative']] = (
    final_result.groupby('Region')[['slavic_name', 'non_slavic_name']]
    .cumsum()
)


final_result.columns.values[1] = 'Date'

In [9]:
final_result = final_result.merge(dep_long,
                                  on = ['Region', 'Date'],
                                  how = 'left')
final_result = final_result.merge(data_cs, on = 'Region', how = 'left')
merged = final_result.merge(
    data_gov,
    on='Region',
    how='left'
)

# Filter rows where the date falls within the governor's term
merged = merged[
    (merged['Date'] >= merged['start']) &
    (merged['Date'] <= merged['end'])
]



In [10]:
merged.to_csv('../data/intermediate/monthly.csv')

## Annual data

In [11]:
daily = pd.read_csv('../data/daily.csv')
daily = daily.rename(columns={'region': "Region"})

daily["total"] = 1
daily["drafted"] = daily["pmc"] = daily["volunteers"] = daily["prisoners"] = daily['contract'] = 0
daily.loc[daily['branch'] == 'добровольцы', 'volunteers'] = 1
daily.loc[daily['branch'] == 'мобилизованные', 'drafted'] = 1
daily.loc[daily['branch'] == 'ЧВК', 'pmc'] = 1
daily.loc[daily['branch'] == 'заключенные', 'prisoners'] = 1
daily.loc[daily['branch'].isin(contract), 'contract'] = 1


In [18]:
df_annual = daily.copy()
df_annual = df_annual[df_annual['death_month'].notna()]
df_annual['year'] = df_annual['death_month'].astype(str).str[-4:].astype(int)
df_annual['non_slavic'] = 1 - df_annual['slavic']
casualty_cols = ['total', 'drafted', 'pmc', 'volunteers', 'prisoners', 'contract', 'slavic', 'non_slavic']
annual = df_annual.groupby(['Region', 'year'])[casualty_cols].sum().reset_index()

region_name_mapping = {
    'Архангельская область': 'Архангельская область без АО',
    'Тюменская область': 'Тюменская область без АО'
}

annual["Region"] = annual["Region"].replace(region_name_mapping)
data_se= pd.read_csv('../data/intermediate/soc_econ_data.csv')
data_se = data_se.drop(columns=["Unnamed: 0"])
data_se['Region'] = data_se['Region'].replace(region_name_mapping)
region_name_mapping = {
    'Архангельская область': 'Архангельская область без АО',
    'Тюменская область': 'Тюменская область без АО'
}
annual = annual.merge(data_se, on = ['Region', 'year'], how = 'left')
annual = annual.merge(data_ex, on = 'Region', how = 'left')
annual

,Region,year,total,drafted,pmc,volunteers,prisoners,contract,slavic,non_slavic,...,disability_per10k,uneployment,median_income,consumption,av_income,share_poverty,% Russians,% Slavs,Excessive Marriages,ex_nup_per100k
0,Алтайский край,2022,400,0,14,33,8,291,376,24,...,63.2,3.7,24037.0,25062.0,31145.0,15.4,95.45,96.08,2007.6,430.387834
1,Алтайский край,2023,677,80,40,120,218,51,631,46,...,NaN,NaN,NaN,NaN,NaN,NaN,95.45,96.08,2007.6,430.387834
2,Алтайский край,2024,985,91,0,425,62,137,921,64,...,NaN,NaN,NaN,NaN,NaN,NaN,95.45,96.08,2007.6,430.387834
3,Алтайский край,2025,30,0,0,28,0,2,28,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2007.6,430.387834
4,Амурская область,2022,100,0,5,25,0,67,91,9,...,39.5,4.2,34565.7,35974.0,44900.0,13.3,95.17,95.95,1890.4,1007.471834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340,Ямало-Ненецкий автономный округ,2025,38,6,0,15,0,1,26,12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
341,Ярославская область,2022,71,0,0,14,36,21,68,3,...,40.5,5.0,30524.1,30810.0,38060.0,8.8,96.52,97.04,780.0,293.731100
342,Ярославская область,2023,194,59,8,44,46,10,190,4,...,NaN,NaN,NaN,NaN,NaN,NaN,96.52,97.04,780.0,293.731100
343,Ярославская область,2024,230,25,5,89,0,35,222,8,...,NaN,NaN,NaN,NaN,NaN,NaN,96.52,97.04,780.0,293.731100


In [19]:
annual.to_csv('../data/intermediate/annual.csv')